# homr — Türk müziği (makam) ince ayarı

SymbTr'den üretilmiş porte verisiyle homr'un **arıza (lift)** ve **ritim** başlarını ince ayarlar.

**Runtime → Change runtime type → GPU** seç. A100 / L4 / V100 ideal (bf16 destekler).

Hücreler sırayla çalıştırılır. 6. hücre kısa deneme, 7. hücre gerçek eğitim.

In [ ]:
#@title 1. GPU uygun mu?
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\nGPU YOK -> Runtime > Change runtime type > GPU')
else:
    bf16 = torch.cuda.is_bf16_supported()
    print('kart:', torch.cuda.get_device_name(0), '| bf16:', bf16)
    print('\nHazir.' if bf16 else
          '\nUYARI: bu kart bf16 desteklemiyor (T4 gibi).'
          ' 6. ve 7. hucrelere --fp32 ekle, yoksa egitim patlar.')

In [ ]:
#@title 2. Depoyu çek
REPO_URL = 'https://github.com/mtalhabalci/homr_tmn.git'  #@param {type:"string"}
BRANCH   = 'makam-finetune'  #@param {type:"string"}

import os, shutil
# Kernel silinecek klasörün içinde duruyorsa git hiçbir şey yapamaz:
# "cannot access parent directories". Önce dışarı çık.
os.chdir('/content')
if os.path.isdir('/content/homr'):
    shutil.rmtree('/content/homr')
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/homr
os.chdir('/content/homr')
!git log --oneline -1


In [ ]:
#@title 3. Bagimliliklar
# transformers 5.x TrainingArguments'tan warmup_ratio'yu kaldirdi;
# depo 4.x bekliyor (pyproject.toml: ^4.53.2), o yuzden sabitliyoruz.
!pip install -q 'transformers>=4.53.2,<5' albumentations editdistance \
                'musicxml==1.4' x-transformers onnxruntime pymupdf

import transformers, inspect
from transformers import TrainingArguments
print('transformers', transformers.__version__)
need = ['warmup_ratio', 'eval_strategy', 'torch_compile', 'bf16']
have = inspect.signature(TrainingArguments.__init__).parameters
for n in need:
    print('  %-16s %s' % (n, 'var' if n in have else 'YOK -> surum uyumsuz'))


## 4. Veri

Porte resimleri git'e sigmayacak kadar buyuk (~250 MB), o yuzden Drive'dan geliyor.

Drive'da **homr_makam** adli bir klasor ac ve **symbtr_veri.tar.gz** dosyasini
icine koy. Arsiv Colab'in *yerel diskine* aciliyor, egitim oradan okuyor -
Drive'dan dogrudan okumak cok yavas olurdu.

Arsivi yeniden uretmek gerekirse, yerel makinede:
`python -m notebooks.package_dataset`


In [ ]:
#@title 4. Veriyi Drive'dan aç
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, tarfile, time
os.chdir('/content/homr')
DRIVE = '/content/drive/MyDrive/homr_makam'
archive = f'{DRIVE}/symbtr_veri.tar.gz'
assert os.path.exists(archive), f'{archive} yok - once Drive a yukle'
t0 = time.time()
with tarfile.open(archive) as t:
    t.extractall('/content/homr')
print('acildi (%.0f sn)' % (time.time() - t0))

# Bölmeler arşivden bağımsız güncellenebilsin diye: Drive'da daha yeni bir
# index dosyası varsa arşivden çıkanın üzerine yazılır.
for name in ('index.txt', 'index_train.txt', 'index_val.txt', 'index_test.txt'):
    source = f'{DRIVE}/{name}'
    if os.path.exists(source):
        shutil.copy(source, f'/content/homr/datasets/SymbTr-2.0.0/{name}')
        print('Drive dan alindi:', name,
              sum(1 for _ in open(source)), 'satir')


In [ ]:
#@title 5. Sözlük ve veri kontrolü (eğitimden önce)
import os, sys
os.chdir('/content/homr'); sys.path.insert(0, '/content/homr')
from homr.transformer.vocabulary import Vocabulary

v = Vocabulary()
print('rhythm %d | lift %d | pitch %d' % (len(v.rhythm), len(v.lift), len(v.pitch)))
print('ariza jetonlari  :', [k for k in v.lift if k[:5] in ('sharp', 'flat')])
print('keyAccidental    :', 'keyAccidental' in v.rhythm)
print('timeSignature_9/8:', 'timeSignature_9/8' in v.rhythm)
print()
for name in ('train', 'val', 'test'):
    path = f'datasets/SymbTr-2.0.0/index_{name}.txt'
    print('%-6s %6d porte' % (name, sum(1 for _ in open(path))))

bad = 0
with open('datasets/SymbTr-2.0.0/index_train.txt') as f:
    rows = [l for l in f if l.strip()][:300]
for row in rows:
    image, tokens = row.strip().split(',')
    if not (os.path.exists(image) and os.path.exists(tokens)):
        bad += 1
        continue
    for line in open(tokens, encoding='utf-8'):
        p = line.split()
        if len(p) == 5 and (p[0] not in v.rhythm or p[2] not in v.lift):
            bad += 1
print('\n300 ornek kontrol edildi, sorunlu:', bad)

In [ ]:
#@title 6. KISA DENEME — 1 epoch, 400 porte
# Ilk calistirmada hazir checkpoint'i indirir (~279 MB).
# bf16 desteklemeyen kartta sona --fp32 ekle.
!cd /content/homr && python -m training.transformer.train --fine --epochs 1 --limit 400

In [ ]:
#@title 7. EĞİTİM + DEĞERLENDİRME — tüm ağ, ~70 dk
# --full: sadece çıkış katmanları değil, bütün ağ eğitilir. Dar kipte ağın
# yalnızca %0,5'i öğreniyordu ve görsel ayrımlar orada takılı kalıyordu.
# Çıktılar tarih damgalı yazılır, eskisinin üstüne yazılmaz.
import glob, os, time
DRIVE = '/content/drive/MyDrive/homr_makam'
DAMGA = time.strftime('%m%d-%H%M')
LOG = f'{DRIVE}/log_{DAMGA}.txt'
print('Log:', LOG)

!cd /content/homr && python -m training.transformer.train --fine --full \
    --work-dir {DRIVE} 2>&1 | tee {LOG}

model = max(glob.glob(f'{DRIVE}/pytorch_model_*.pth'), key=os.path.getmtime)
OUT = f'{DRIVE}/eval_{DAMGA}.txt'
print('\nDegerlendirilen model:', model)
print('Sonuc:', OUT)
!cd /content/homr && python -m training.evaluate_makam \
    --checkpoint "{model}" 2>&1 | tee {OUT}


In [ ]:
#@title 8. Değerlendirme — sınıf sınıf
MODEL = ''  #@param {type:"string"}
# Boş bırakırsan Drive'daki en yeni .pth seçilir. Aşağıdaki listede yanlış
# olanı seçtiğini görürsen doğru dosyanın tam yolunu buraya yapıştır.

import glob, os, time
DRIVE = '/content/drive/MyDrive/homr_makam'
adaylar = sorted(glob.glob(f'{DRIVE}/*.pth'), key=os.path.getmtime, reverse=True)
assert adaylar, f'{DRIVE} altinda .pth yok'
print(f"{'':2}{'dosya':<52}{'MB':>7}  degistirilme")
for i, yol in enumerate(adaylar):
    isaret = '->' if i == 0 else '  '
    print(f'{isaret}{os.path.basename(yol)[:52]:<52}'
          f'{os.path.getsize(yol)/1e6:>7.0f}  '
          f"{time.strftime('%d.%m %H:%M', time.localtime(os.path.getmtime(yol)))}")

model = MODEL if MODEL else adaylar[0]
# Tarih damgası: her koşu kendi dosyasına yazsın, eskisi kaybolmasın.
OUT = f"{DRIVE}/eval_{time.strftime('%m%d-%H%M')}.txt"
print('\nSecilen:', model)
print('Sonuc  :', OUT)
!cd /content/homr && python -m training.evaluate_makam \
    --checkpoint "{model}" 2>&1 | tee {OUT}


In [ ]:
#@title 9. Başlangıç noktası — ham homr, ince ayarsız
# Aynı sınav, ince ayardan ÖNCEKİ model. Makam jetonları sözlüğünde yok,
# o yüzden --lenient: Batı diyezi sharp4, Batı bemolü flat5 sayılıyor.
import os, sys, time
os.chdir('/content/homr')
sys.path.insert(0, '/content/homr')
from homr.transformer.configs import Config
from training.transformer.train import download_training_checkpoint

yapilandirma = Config()
download_training_checkpoint(yapilandirma)
HAM = yapilandirma.filepaths.checkpoint
assert os.path.exists(HAM), f'indirilemedi: {HAM}'
print('Ham model:', HAM, f'({os.path.getsize(HAM)/1e6:.0f} MB)')

DRIVE = '/content/drive/MyDrive/homr_makam'
OUT = f"{DRIVE}/baseline_{time.strftime('%m%d-%H%M')}.txt"
print('Sonuc:', OUT)
!cd /content/homr && python -m training.evaluate_makam \
    --checkpoint "{HAM}" --lenient 2>&1 | tee {OUT}


In [ ]:
#@title 10. Serbest okuma — model kendi başına okuyor
# Diğer ölçümlerde modele her adımda önceki DOĞRU sembol veriliyor.
# Burada porteyi baştan sona kendi okuyor, iki dizi düzenleme mesafesiyle
# hizalanıyor. Gerçek kullanımdaki performans bu. Porte porte gittiği için
# yavaş: ~10-15 dakika.
MODEL = ''  #@param {type:"string"}

import glob, os, time
DRIVE = '/content/drive/MyDrive/homr_makam'
model = MODEL if MODEL else max(
    glob.glob(f'{DRIVE}/pytorch_model_*.pth'), key=os.path.getmtime)
OUT = f"{DRIVE}/serbest_{time.strftime('%m%d-%H%M')}.txt"
print('Model :', model)
print('Sonuc :', OUT)
!cd /content/homr && python -m training.evaluate_makam \
    --checkpoint "{model}" --generate 2>&1 | tee {OUT}


In [ ]:
#@title 11. Tek eseri okut — gösterim için
ESER = 'rast--medhal--hafif--musahabet_i_musikiye--refik_fersan'  #@param {type:"string"}
MODEL = ''  #@param {type:"string"}
# PDF'i Drive'da homr_makam/pdf/ altina koy. Veri setinde OLMAYAN eserler de
# okunabilir: porte kesme usulden bagimsiz calisiyor.

import glob, os, shutil
DRIVE = '/content/drive/MyDrive/homr_makam'
HEDEF = '/content/homr/datasets/SymbTr-2.0.0/pdf'
os.makedirs(HEDEF, exist_ok=True)
kaynak = f'{DRIVE}/pdf/{ESER}.pdf'
assert os.path.exists(kaynak), f'{kaynak} yok - PDF i Drive a yukle'
shutil.copy(kaynak, f'{HEDEF}/{ESER}.pdf')

model = MODEL if MODEL else max(
    glob.glob(f'{DRIVE}/pytorch_model_*.pth'), key=os.path.getmtime)
CIKTI = f'{DRIVE}/gosterim'
print('Model :', model)
print('Cikti :', CIKTI)
!cd /content/homr && python -m training.read_work \
    --work "{ESER}" --checkpoint "{model}" --out "{CIKTI}"
